# 4. Capstone project: your own debt thesis

Unit: America's Debt Crisis. Concept level: **understand → analyze → challenge → capstone**.

Data: the course dataset lives in one deterministic DuckDB file, `../../data/analytics.duckdb`, seeded
from `seed/seed.sql` (identical inside the Docker CLI). Every number here is
stable: rerun any notebook years from now and it reproduces the same answers,
because the seed uses fixed anchors (the course video's figures) plus
deterministic noise -- never the random number generator.

Workflow: run cells top to bottom. A `# TASK` comment marks a cell you should
edit; answer in the markdown cell just below when asked.

Mission (capstone level): **choose one scenario, load the dataset, refit a
chart, and defend a verdict** -- the exact move the video makes, with your own
numbers.

Pick **one**:
* **A — "The market is the truth"**: show g - r under the 5.3% 30-year, and
  defend whether the ~2038 (or never) crossover is the real price.
* **B — "Growth is the medicine"**: the 333 plan with 3% + 3% + 3M -- prove
  the answer still hinges on the primary deficit, not the interest bill.
* **C — "The interest hurricane"**: replay `02` but make the stress a *path*:
  higher in 2026-2030, fading after. Find the crossover.

Deliverable: a working cell + a written defense under `DEFENSE`. Graders value
(1) a clear hypothesis, (2) a reproducible cell, (3) a defense that names the
*lever* that would change your answer.

### 4.1 Boot

In [ ]:
# Bootstrap
%matplotlib inline
import sys
from pathlib import Path
import duckdb

sys.path.insert(0, "../../scripts")   # the g-vs-r model used in 02-03
import model as m

DB = "../../data/analytics.duckdb"
con = duckdb.connect(DB)

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"figure.figsize": (8, 4.4),
                     "axes.grid": True, "grid.alpha": 0.35,
                     "font.size": 10})

In [ ]:
print(con.execute(
    "SELECT year, debt_total_billions, debt_held_public_pct_gdp "
    "FROM debt ORDER BY year").fetchdf())

### 4.2 Your scenario

In [ ]:
# TASK: pick A, B, or C
SCENARIO = "A"   # <- change me

if SCENARIO == "A":
    rows = m.project(50, r=round(m.BASELINE_R + m.STATIC_R_BP[2] / 100, 3))
elif SCENARIO == "B":
    rows = m.project(50, g2026=3.0)
else:
    rows = m.project(50)
df = pd.DataFrame(rows)
print("SCENARIO:", SCENARIO, " crossover:", m.crossover(rows))

In [ ]:
fig, ax = plt.subplots()
ax.plot(df.year, df.g, label="g", color="C0")
ax.plot(df.year, df.r, label="r", color="C3")
ax.plot(df.year, df.interest_pct_gdp, ls="--", color="k", label="interest % GDP")
c = m.crossover(rows)
if c:
    ax.axvline(c, color="C2", ls=":", lw=1)
    ax.text(c, df.g.min(), "crossover", color="C2", ha="center")
ax.legend(); ax.set(xlabel="year", ylabel="%/year",
                    title=f"Scenario {SCENARIO}")
plt.show()

### 4.3 Sensitivity your way

In [ ]:
# TASK: which lever moves IT?  (r = market rate, g = growth healing)
lever = "r"
if lever == "r":
    table = m.sensitivity()
else:
    table = []
    for step in (0.26, 0.35, 0.5, 0.75):
        m.GROWTH_STEP = step
        table.append({"step": step, "crossover": m.crossover(m.project(50))})
    m.GROWTH_STEP = 0.26
print(pd.DataFrame(table).to_string(index=False))

### 4.4 Defense

> **DEFENSE** — write 3-5 sentences here. Name: (a) your scenario, (b) the
> *lever* that matters most to your answer, (c) the single number that would
> make you change your mind, (d) a counterargument. Then run
> **Kernel -> Restart & Run All** to prove reproducibility.

---
End of the unit. Debrief with `lesson_plans.md`, `exercises.md`, `solutions.md`.